In [25]:
from dotenv import load_dotenv
import pymupdf4llm
import tiktoken
import json
import os
from markdown_it import MarkdownIt
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.language_models import BaseChatModel
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
import textwrap
from typing import Any

load_dotenv()

REDO_ALL = True

In [26]:
FILE_NAME = "AI Transformation_KickoffSlide_v1.0.pdf"
DATA_PATH = f"../data/{FILE_NAME}"
OUTPUT_PATH = f"../output/{FILE_NAME}"

md = pymupdf4llm.to_markdown(DATA_PATH)

if not os.path.exists("../output"):
    os.makedirs("../output")

with open(OUTPUT_PATH, "w") as f:
    f.write(md)

=== Document parser messages ===
                                    Full-page OCR on page.number=34/35.



In [27]:
# json_str = pymupdf4llm.to_json(data_path)

# dict_data = json.loads(json_str)

# with open("../output/Group_Project.json", "w") as f:
#     json.dump(dict_data, f, indent=4)

In [28]:
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    tokens = encoding.encode(text)
    return len(tokens)

token_count = count_tokens(md)
token_count

7676

In [29]:
def parse_markdown_tree(markdown_text: str):
    md = MarkdownIt()
    tokens = md.parse(markdown_text)

    root = {"title": "root", "level": 0, "children": []}
    stack = [root]

    i = 0
    while i < len(tokens):
        token = tokens[i]

        if token.type == "heading_open":
            level = int(token.tag[1])  # h1 -> 1, h2 -> 2

            inline = tokens[i + 1]
            title = inline.content

            node = {
                "title": title,
                "level": level,
                "children": []
            }

            # find correct parent
            while stack and stack[-1]["level"] >= level:
                stack.pop()

            stack[-1]["children"].append(node)
            stack.append(node)

        i += 1

    return root["children"]

markdown_tree = parse_markdown_tree(md)
markdown_tree[:2]

tree_token_count = count_tokens(json.dumps(markdown_tree))
print(f"Markdown tree token count: {tree_token_count}")

with open("../output/markdown_tree.json", "w") as f:
    json.dump(markdown_tree, f, indent=4)

Markdown tree token count: 2436


In [30]:
headers_to_split_on = [
    ("#", "header_1"),
    ("##", "header_2"),
    ("###", "header_3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(md)
md_header_splits[:5]

[Document(metadata={'header_1': '**AI TRANSFORMATION**'}, page_content='**Kickoff Meeting**  \nMar 2026  \nCopyright @ 2026 TECHVIFY | A Global AI & Software Solutions Company Follow Us – Website | LinkedIn | Twitter | Facebook  \n1  \n**==> picture [30 x 31] intentionally omitted <==**  \n02  \n**==> picture [39 x 31] intentionally omitted <==**  \n**==> picture [41 x 30] intentionally omitted <==**  \n**==> picture [39 x 30] intentionally omitted <==**  \n**==> picture [39 x 30] intentionally omitted <==**  \nTECHVIFY @ 2026 All Rights Reserved.  \nTECHVIFY @ 2026. All Rights Reserved.'),
 Document(metadata={'header_1': '**AI TRANSFORMATION**', 'header_2': '**Business Goals**'}, page_content='Transforming our delivery center from traditional software engineering to an AI-Driven Development model  \n**01 02 03 Boost Engineer Raise AI Productivity Upskilling Maturity**  \n**04 05 Human + AI AI Across Teamwork Full SDLC**  \n- 3–5× faster delivery  \n- AI-assisted coding  \n- Auto test 

In [31]:
class H4Header(BaseModel):
    title: str
    """
    The small header title, which should be the text of the header that defines the small header (e.g., the text of an H4 header for a small header defined by an H4 header).
    """

class Subsection(BaseModel):
    title: str
    """
    The subsection title, which should be the text of the header that defines the subsection (e.g., the text of an H3 header for a subsection defined by an H3 header).
    """
    h4_headers: list[H4Header]
    """
    The H4 headers of the subsection, where each H4 header can have its own title. The structure should reflect the hierarchy of the document based on the headers (e.g., H4 headers would define small headers within an H3 subsection).
    """ 

    def get_h4_header_titles(self) -> list[str]:
        """
        Get a list of all H4 header titles in the subsection.

        Returns:
            A list of H4 header titles.
        """
        return [f"{self.title}: {h4_header.title}" for h4_header in self.h4_headers]

class Section(BaseModel):
    title: str
    """
    The section title, which should be the text of the header that defines the section (e.g., the text of an H2 header for a section defined by an H2 header).
    """
    subsections: list[Subsection]
    """
    The subsections of the section, where each subsection can have its own title and content. The structure should reflect the hierarchy of the document based on the headers (e.g., H3 headers would define subsections within an H2 section).
    """

    def get_subsection_titles(self) -> list[str]:
        """
        Get a list of all subsection titles in the section.

        Returns:
            A list of subsection titles.
        """
        return [f"{self.title}: {subsection.title}" for subsection in self.subsections]

class DocumentStructure(BaseModel):
    title: str
    """
    The document title, which should be the text of the first header in the document (e.g., the text of the first H1 header).
    """
    sections: list[Section]
    """
    The sections of the document, where each section can have subsections. The structure should reflect the hierarchy of the document based on the headers.
    """

    def get_section_titles(self) -> list[str]:
        """
        Get a list of all section titles in the document.

        Returns:
            A list of section titles.
        """
        return [f"{self.title}: {section.title}" for section in self.sections]
    
    def get_document_tree(self, level: int = 3) -> str:
        """
        Get the document structure tree as a folder-style structure.

        Args:
            level: The level of the tree to generate (1 for title only, 2 for sections, 3 for subsections, 4 for H4 headers).
        Returns:
            A string representing the document structure tree in a folder-style format.
        """
        if level < 1 or level > 4:
            raise ValueError("Level must be between 1 and 4")

        def build_tree(node: dict[str, Any], prefix: str = "", is_last: bool = True, current_level: int = 1) -> str:
            connector = "└── " if is_last else "├── "
            tree_str = prefix + connector + node["title"] + "\n"

            if current_level >= level:
                return tree_str

            children = node.get("children", [])
            if not children:
                return tree_str

            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, child in enumerate(children):
                last_child = i == len(children) - 1
                tree_str += build_tree(child, new_prefix, last_child, current_level + 1)

            return tree_str

        root: dict[str, Any] = {
            "title": self.title,
            "children": []
        }

        for section in self.sections:
            section_node: dict[str, Any] = {
                "title": section.title,
                "children": []
            }

            for subsection in section.subsections:
                subsection_node: dict[str, Any] = {
                    "title": subsection.title,
                    "children": []
                }

                for h4_header in subsection.h4_headers:
                    h4_node: dict[str, Any] = {
                        "title": h4_header.title
                    }
                    subsection_node["children"].append(h4_node)

                section_node["children"].append(subsection_node)

            root["children"].append(section_node)

        # root printed without connector
        tree = root["title"] + "\n"
        if level > 1:
            for i, child in enumerate(root["children"]):
                tree += build_tree(child, "", i == len(root["children"]) - 1, current_level=2)

        return tree

In [32]:
# Document structure resolver
class DocumentStructureResolver:
    PROMPT = textwrap.dedent("""
    Given a list of headers with their levels and texts, determine the hierarchical structure of the document. 
    The first header should be considered the document title, and subsequent headers should be organized into sections and subsections based on their levels. 
    For example, an H2 header would define a section, and an H3 header would define a subsection within the most recent H2 section. 
    The output should be a structured representation of the document's hierarchy, including the document title, sections, and subsections.
    Note that the input will be a list of dictionaries, where each dictionary contains metadata about a header (e.g., header level and text).
    The headers within this input may be incorrect and do not properly represent the document structure, 
    so you should resolve the correct structure based on the header contents instead of relying solely on the header levels.
    """)
    def __init__(self, llm: BaseChatModel):
        self.llm = llm.with_structured_output(DocumentStructure)

    def resolve_structure(self, headers: list[dict[str, str]]) -> DocumentStructure:
        """
        Resolve the document structure based on the provided headers.

        Args:
            headers: A list of dictionaries, where each dictionary contains metadata about a header (e.g., header level and text).

        Returns:
            A DocumentStructure object representing the hierarchical structure of the document.
        """
        response = self.llm.invoke(
            [
                {
                    "role": "system",
                    "content": self.PROMPT
                },
                {
                    "role": "user",
                    "content": str(headers)
                }
            ]
        )
        if not isinstance(response, DocumentStructure):
            raise ValueError("Expected response to be of type DocumentStructure")
        
        return response
    
llm = ChatOpenAI(model="gpt-4.1")

structure_resolver = DocumentStructureResolver(llm)

if REDO_ALL or not os.path.exists("../output/document_structure.json"):
    document_structure = structure_resolver.resolve_structure(markdown_tree)

    with open("../output/document_structure.json", "w") as f:
        json.dump(document_structure.model_dump(), f, indent=4)
else:
    with open("../output/document_structure.json", "r") as f:
        document_structure_dict = json.load(f)
        document_structure = DocumentStructure.model_validate(document_structure_dict)

In [33]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = InMemoryVectorStore(embeddings)

def create_batches(items: list[Document], max_tokens: int) -> list[list[Document]]:
    batches: list[list[Document]] = []
    current_batch: list[Document] = []
    current_tokens = 0

    for item in items:
        item_tokens = count_tokens(item.page_content)

        if current_tokens + item_tokens > max_tokens:
            batches.append(current_batch)
            current_batch = []
            current_tokens = 0

        current_batch.append(item)
        current_tokens += item_tokens

    if current_batch:
        batches.append(current_batch)

    return batches

batches = create_batches(md_header_splits, max_tokens=8000)

print(f"Number of batches: {len(batches)}")

ids: list[str] = []

for i, batch in enumerate(batches):
    print(f"Batch {i + 1} token count: {count_tokens(' '.join([doc.page_content for doc in batch]))}")
    batch_ids = vector_store.add_documents(batch)
    ids.extend(batch_ids)

ids[:3]

Number of batches: 1
Batch 1 token count: 6665


['61bc0bae-5483-4bd6-bffb-c65e38321020',
 '5bcdde5e-c6e9-4202-bb4e-d31351afd719',
 '60064f85-27d3-41cf-8a4d-775f2d04d287']

In [34]:
import numpy as np

def elbow_filter(data: list[tuple[Document, float]], max_k: int = 10) -> list[tuple[Document, float]]:
    """
    Determine the optimal top_k for similarity search results using the elbow method.

    Args:
        data: A list of tuples, where each tuple contains a Document and its relevance score.
        max_k: The maximum number of clusters to test for the elbow method.
    Returns:
        A list of tuples containing the filtered Document and relevance score pairs.
    """
    # Extract relevance scores from the data
    relevance_scores = np.array([score for _, score in data])

    # Calculate the sum of squared distances for different values of k
    ssd = []
    for k in range(1, max_k + 1):
        # Perform k-means clustering on the relevance scores
        from sklearn.cluster import KMeans

        kmeans = KMeans(n_clusters=k, random_state=0).fit(relevance_scores.reshape(-1, 1))
        ssd.append(kmeans.inertia_)

    # Determine the optimal k using the elbow method
    optimal_k = 1
    for i in range(1, len(ssd) - 1):
        if (ssd[i - 1] - ssd[i]) / (ssd[i] - ssd[i + 1]) > 1.5:  # Elbow point heuristic
            optimal_k = i + 1
            break

    # Filter the data to keep only the top optimal_k results
    filtered_data = data[:optimal_k]
    return filtered_data

In [35]:
def retrieve(doc_struct: DocumentStructure, level: int) -> dict[str, list[Document]]:
    """
    Retrieve relevant documents from the vector store based on the provided document structure.

    Args:
        doc_struct: A DocumentStructure object representing the hierarchical structure of the document.
        level: The level of the document structure to retrieve (e.g., 1 for sections, 2 for subsections).
    Returns:
        A dictionary mapping section/subsection titles to lists of relevant Documents retrieved from the vector store.
    """
    if level > 3 or level < 1:
        raise ValueError("Level must be between 1 and 3")

    titles: list[tuple[str, str]] = []  # (original_title, modified_title)

    if level == 1:
        modified_titles = doc_struct.get_section_titles()
        for section, modified in zip(doc_struct.sections, modified_titles):
            titles.append((section.title, modified))

    elif level == 2:
        for section in doc_struct.sections:
            modified_titles = section.get_subsection_titles()
            for subsection, modified in zip(section.subsections, modified_titles):
                titles.append((subsection.title, modified))

    elif level == 3:
        for section in doc_struct.sections:
            for subsection in section.subsections:
                modified_titles = subsection.get_h4_header_titles()
                for h4_header, modified in zip(subsection.h4_headers, modified_titles):
                    titles.append((h4_header.title, modified))

    retrieved_docs: dict[str, list[Document]] = {}

    for title in titles:
        results = vector_store.similarity_search_with_score(title[1], k=20)  # returns list of tuples (Document, relevance_score)
        filtered_results = elbow_filter(results)
        docs = [doc for doc, _ in filtered_results]
        retrieved_docs[title[0]] = docs

    return retrieved_docs

retrieved_docs = retrieve(document_structure, level=2)
retrieved_docs

{'The Typist': [Document(id='9398e1fa-4212-432a-be64-785083b5708a', metadata={'header_1': '**AI TRANSFORMATION**', 'header_2': '**Orchestration  •  Direction  •  Judgment**'}, page_content='_"We took product managers, designers, frontend engineers, and backend engineers... combined them... they\'re all full-stack builders."_  \n— Satya Nadella, CEO Microsoft (Davos, January 2026)  \n_”As you step into the role of an orchestrator, move your strength from handson coding to strategic thinking. Instead of fixing the code, refine the AI to unlock greater efficiency."_  \n— David Ho, Head of Technology Consulting (Techvify HQ, Mar 2026)  \n6'),
  Document(id='5b38e5ca-ca9b-470c-9e93-ba96879838cc', metadata={'header_1': '**IC Maturity Level**', 'header_2': '**List of AI Champions**'}, page_content='|#|Delivery|AI Champions|Job Title|\n|---|---|---|---|\n|1|D1|Andy.Le|Software Engineer|\n|2||Andy.Hoang|Software Engineer|\n|3||Fred.Nguyen|Software Engineer|\n|4|D5|Thang Nguyen|Senior Software E

In [37]:
document_tree = document_structure.get_document_tree()
print(document_tree)

AI TRANSFORMATION
├── Business Goals
├── Transition from Traditional Engineering to AI Augmented Engineering
├── One pizza team of 3-5 people
├── Two pizza team of 8-10 people
├── Platform Team Layer
├── Guardrails
├── Standard Infrastructure
├── Compliance as Code
├── Full-Stack Builder: From Coder to Orchestrator (New Techvifer NDA)
│   ├── The Typist
│   ├── The OODA Loop
│   ├── The Orchestrator
│   └── Orchestration  •  Direction  •  Judgment
├── Skills That Matter Now vs Skills AI Handles
├── IC Maturity Level
│   ├── Skills Progression Matrix
│   ├── Role
│   ├── Level 1: AI Assisted (Beginner)
│   ├── Level 2: AI Driven (Practitioner)
│   ├── Level 3: AI Native (Expert)
│   ├── Developer
│   ├── Architect
│   └── QA Engineer
├── Company Maturity Level
│   └── Next in 3 – 6 Months
├── Security & Compliance
│   ├── AI Threat Landscape
│   ├── Governance & Mitigation
│   ├── Prompt Injection Attacks
│   ├── Prompt Firewall & Input Validation
│   ├── MCP Server Exploitation
│   ├──

In [38]:
retrieved_docs_token_counts = {title: count_tokens(doc.page_content) for title, docs in retrieved_docs.items() for doc in docs}
total_tokens = sum(retrieved_docs_token_counts.values())
print(retrieved_docs_token_counts)
print(f"Total tokens in retrieved documents: {total_tokens}")
print(f"Efficiency: {total_tokens / token_count:.2%} of original document tokens")

{'The Typist': 196, 'The OODA Loop': 389, 'The Orchestrator': 389, 'Orchestration  •  Direction  •  Judgment': 389, 'Skills Progression Matrix': 389, 'Role': 11, 'Level 1: AI Assisted (Beginner)': 307, 'Level 2: AI Driven (Practitioner)': 11, 'Level 3: AI Native (Expert)': 11, 'Developer': 389, 'Architect': 389, 'QA Engineer': 324, 'Next in 3 – 6 Months': 74, 'AI Threat Landscape': 44, 'Governance & Mitigation': 12, 'Prompt Injection Attacks': 14, 'Prompt Firewall & Input Validation': 14, 'MCP Server Exploitation': 14, 'MCP Zero-Trust Architecture': 16, 'Data Poisoning & Model Tampering': 16, 'Model Security & Supply Chain Audit': 15, 'AI Supply Chain Vulnerabilities': 44, 'AI Access Control & Data Governance': 44, 'Sensitive Data Leakage & Hallucination': 16, 'Output Guardrails & Compliance': 16, 'Program Team Structure': 324, 'TC Team': 6, 'All Delivery Units': 6, 'PQA Team': 169, 'Support & Tracking': 169, 'Four-Phase Rollout Strategy': 183, 'Launch': 183, 'Phase 2': 6, 'Deepening':

In [39]:
class Summarizer:
    PROMPT = textwrap.dedent("""
    Given a set of relevant documents retrieved based on the document structure, summarize the content of these documents in a way that captures the key information while being concise. 
    The summary should be structured according to the document hierarchy, with clear indications of which sections and subsections the summarized content corresponds to. 
    The summary should provide a comprehensive overview of the main points and key information from the retrieved documents, while being concise and easy to understand. 
    The summary should be organized in a way that reflects the structure of the original document, with clear headings and subheadings corresponding to the sections and subsections of the document.
    These are the expected layout of the summarization:
    Title: [Document Title]
    Description: [A brief description of the document, including its purpose and main topics covered]
    Sections:
    - [Section Title 1]: [A concise summary of the content in this section, highlighting the key points and information]
    - [Section Title 2]: [A concise summary of the content in this section, highlighting the key points and information]
    - ... And so on for each section and subsection, organized according to the document structure.
    """)
    def __init__(self, llm: BaseChatModel):
        self.llm = llm

    def summarize(self, relevant_docs: dict[str, Document], doc_structure: DocumentStructure, level: int = 3) -> str:
        """
        Summarize the relevant documents based on the provided document structure.

        Args:
            relevant_docs: A dictionary mapping section/subsection titles to the most relevant Document retrieved from the vector store.
            doc_structure: A DocumentStructure object representing the hierarchical structure of the document.
            level: The level of the document structure to include in the summary (e.g., 1 for sections, 2 for subsections).
        Returns:
            A string containing the structured summary of the relevant documents.
        """
        level_to_prompt = {
            1: "High-level",
            2: "Mid-level",
            3: "Low-level",
        }
        response = self.llm.invoke(
            [
                {
                    "role": "system",
                    "content": self.PROMPT
                },
                {
                    "role": "user",
                    "content": textwrap.dedent(
                        f"""Relevant documents: {relevant_docs}
                        ---
                        Document structure to be summarized: {doc_structure.get_document_tree(level)}
                        ---
                        The summary should be a {level_to_prompt.get(level, 'Mid-level')} summary based on the document structure provided.
                        """)
                }
            ]
        )

        if not isinstance(response.content, str):
            raise ValueError(f"Expected response to be a string, but got {type(response.content)}")

        return str(response.content)

summarizer = Summarizer(llm)
summary = summarizer.summarize(retrieved_docs, document_structure)
print(summary)

Title: AI TRANSFORMATION

Description:  
This document provides a comprehensive framework for the enterprise-wide transformation from traditional engineering to AI-augmented software delivery. It outlines the required skills, roles, team structures, maturity models, governance, security, operational strategies, rollout plans, KPIs, and commercial ramifications to ensure sustainable competitive advantage and measurable business value.

---

Sections:

**Business Goals**:  
Outlines objectives for increasing delivery efficiency, quality, and market differentiation through adoption of AI-powered practices and automation across software lifecycle stages.

**Transition from Traditional Engineering to AI Augmented Engineering**:  
Describes the organizational shift from conventional development to AI-augmented workflows where humans orchestrate and validate AI-driven outputs, emphasizing the need for mindset and culture change.

**One pizza team of 3–5 people / Two pizza team of 8–10 people*

In [40]:
summary_token_count = count_tokens(summary)
print(f"Summary token count: {summary_token_count}")

Summary token count: 2105


In [42]:
class SummarizationPipeline:
    def __init__(self, structure_resolver: DocumentStructureResolver, summarizer: Summarizer, vector_store: InMemoryVectorStore):
        self.structure_resolver = structure_resolver
        self.summarizer = summarizer
        self.vector_store = vector_store

    def run(self, file_path: str, level: int) -> str:
        md = pymupdf4llm.to_markdown(file_path)
        markdown_tree = parse_markdown_tree(md)
        document_structure = self.structure_resolver.resolve_structure(markdown_tree)
        retrieved_docs = retrieve(document_structure, level)
        summary = self.summarizer.summarize(retrieved_docs, document_structure, level)
        return summary
    
pipeline = SummarizationPipeline(structure_resolver, summarizer, vector_store)
final_summary = pipeline.run(DATA_PATH, level=3)
print(final_summary)

=== Document parser messages ===
                                                                        Full-page OCR on page.number=34/35.

Title: AI TRANSFORMATION  
Description:  
This document outlines the transformation journey from traditional engineering to AI-augmented software delivery, including changes in roles, process maturity, security, compliance, program structure, rollout strategies, operating models, and the impact on business and client engagement. The focus is on achieving higher efficiency and quality while managing risks and differentiating through AI-led delivery capabilities.

---

Sections:

### Business Goals
Defines the motivation for AI transformation, centering on increased delivery velocity, higher quality, reduced defects, and improved client outcomes.

---

### Transition from Traditional Engineering to AI Augmented Engineering
- **One pizza team of 3–5 people / Two pizza team of 8–10 people**: Moves away from siloed pods toward lean, integrated teams u

In [43]:
total_summary_tokens = count_tokens(final_summary)
total_summary_tokens

1319

In [44]:
with open("../output/final_summary.md", "w") as f:
    f.write(final_summary)